<a href="https://colab.research.google.com/github/ar2nas/GEPA_DSPY_supernova/blob/main/main_GEPA_Dspy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip install -q -U \
    "dspy==3.2.1" \
    "groq>=1.5,<2" \
    "pandas==2.2.2" \
    "numpy>=1.26,<2.1" \
    "scipy>=1.12,<2" \
    "tqdm>=4.66,<5"

import sys
import importlib.metadata as metadata
import dspy
import groq
import numpy as np
import pandas as pd
import scipy

required_dspy_objects = ["LM", "Example", "Prediction", "Signature", "Module", "GEPA"]
missing = [name for name in required_dspy_objects if not hasattr(dspy, name)]

assert not missing, f"Missing DSPy objects: {missing}"

print(" import dspy completed without errors")
print(f" Python: {sys.version.split()[0]}")
print(f" DSPy: {metadata.version('dspy')}")
print(f" Groq SDK: {metadata.version('groq')}")
print(f" NumPy: {np.__version__}")
print(f" andas: {pd.__version__}")
print(f" SciPy: {scipy.__version__}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 48.3 MB/s eta 0:00:00
 import dspy completed without errors
 Python: 3.12.13
 DSPy: 3.2.1
 Groq SDK: 1.5.0
 NumPy: 2.0.2
 andas: 3.0.3
 SciPy: 1.18.0


In [ ]:


import os
import dspy
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY was not found. In Colab, open Secrets, add "
        "`GROQ_API_KEY`, and enable notebook access."
    )

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

TASK_MODEL_NAME = "groq/llama-3.3-70b-versatile"
JUDGE_MODEL_NAME = "groq/llama-3.3-70b-versatile"
REFLECTION_MODEL_NAME = "groq/llama-3.3-70b-versatile"

task_lm = dspy.LM(
    model=TASK_MODEL_NAME,
    api_key=GROQ_API_KEY,
    temperature=0.2,
    max_tokens=1024,
    cache=False,
)

judge_lm = dspy.LM(
    model=JUDGE_MODEL_NAME,
    api_key=GROQ_API_KEY,
    temperature=0.7,
    max_tokens=1200,
    cache=False,
)

reflection_lm = dspy.LM(
    model=REFLECTION_MODEL_NAME,
    api_key=GROQ_API_KEY,
    temperature=0.8,
    max_tokens=2048,
    cache=False,
)

# The translation/student program uses task_lm by default.
dspy.configure(lm=task_lm)

test_response = task_lm(
    "Reply with exactly: DSPy Groq connection successful."
)

assert test_response, "The Groq model returned an empty response."

print("✅Groq API connection validated")
print(f"Task LM: {TASK_MODEL_NAME}")
print(f"Judge LM: {JUDGE_MODEL_NAME}")
print(f"Reflection LM: {REFLECTION_MODEL_NAME}")
print(test_response)


✅Groq API connection validated
Task LM: groq/llama-3.3-70b-versatile
Judge LM: groq/llama-3.3-70b-versatile
Reflection LM: groq/llama-3.3-70b-versatile
['DSPy Groq connection successful.']


In [ ]:


import re
import dspy
from dspy import Signature, InputField, OutputField, ChainOfThought, Module


class TranslationJudgeSignature(Signature):
    """
    Evaluate a translation against the source and a human reference.
    Score accuracy, fluency, terminology, target-language authenticity,
    and cultural appropriateness.
    """

    source_text = InputField(desc="Original source-language text")
    translated_text = InputField(desc="Candidate translation to evaluate")
    reference_text = InputField(desc="Human reference translation")
    source_lang = InputField(desc="Source language")
    target_lang = InputField(desc="Target language, Cantonese or Mandarin")

    score = OutputField(desc="One numeric quality score from 0 to 10")
    feedback = OutputField(
        desc=(
            "Written diagnostic feedback explaining strengths, errors, "
            "and a concrete improvement. For Cantonese, explicitly check "
            "genuine Cantonese vocabulary and particles rather than "
            "Mandarin written in Traditional Chinese."
        )
    )


class TranslationJudge(Module):
    def __init__(self):
        super().__init__()
        self.predictor = ChainOfThought(TranslationJudgeSignature)

    def forward(
        self,
        source_text,
        translated_text,
        reference_text,
        source_lang="English",
        target_lang="Cantonese",
    ):
        # Keep the judge independent from the student/task LM configuration.
        with dspy.context(lm=judge_lm):
            result = self.predictor(
                source_text=source_text,
                translated_text=translated_text,
                reference_text=reference_text,
                source_lang=source_lang,
                target_lang=target_lang,
            )

        score_match = re.search(r"-?\d+(?:\.\d+)?", str(result.score))

        if not score_match:
            raise ValueError(f"Judge returned a non-numeric score: {result.score!r}")

        result.score = max(0.0, min(10.0, float(score_match.group(0))))
        result.feedback = str(result.feedback).strip()

        if not result.feedback:
            raise ValueError("Judge returned empty written feedback.")

        return result


judge = TranslationJudge()
print("Real Llama translation judge created")


Real Llama translation judge created


In [ ]:

import pandas as pd
import time

print("📝 Testing judge on 5 English-Cantonese examples...")
print("=" * 80)

en_cantonese_examples = [
    {
        "source": "Hello, how are you today?",
        "translation": "你好，今日點樣呀？",
        "reference": "你好，今日點樣呀？",
        "source_lang": "English",
        "target_lang": "Cantonese"
    },
    {
        "source": "I would like to order a cup of coffee.",
        "translation": "我想叫一杯咖啡。",
        "reference": "我想叫一杯咖啡。",
        "source_lang": "English",
        "target_lang": "Cantonese"
    },
    {
        "source": "The weather is beautiful today.",
        "translation": "今日天氣好靚。",
        "reference": "今日天氣好靚。",
        "source_lang": "English",
        "target_lang": "Cantonese"
    },
    {
        "source": "Can you recommend a good restaurant?",
        "translation": "你可唔可以推薦一間好嘅餐廳？",
        "reference": "你可唔可以推薦一間好嘅餐廳？",
        "source_lang": "English",
        "target_lang": "Cantonese"
    },
    {
        "source": "I'm going to the market to buy some vegetables.",
        "translation": "我去街市買啲菜。",
        "reference": "我去街市買啲菜。",
        "source_lang": "English",
        "target_lang": "Cantonese"
    }
]

results_cantonese = []
for idx, example in enumerate(en_cantonese_examples, 1):
    print(f"\n📌 Example {idx}: English → Cantonese")
    print(f"   Source: {example['source']}")
    print(f"   Translation: {example['translation']}")

    try:
        result = judge(
            source_text=example['source'],
            translated_text=example['translation'],
            reference_text=example['reference'],
            source_lang=example['source_lang'],
            target_lang=example['target_lang']
        )

        print(f"   ✅ Score: {result.score:.1f}/10")
        print(f"   📝 Feedback: {result.feedback[:100]}...")

        results_cantonese.append({
            'example': idx,
            'source': example['source'],
            'translation': example['translation'],
            'score': result.score,
            'feedback': result.feedback
        })

    except Exception as e:
        print(f"   ❌ Error: {str(e)}")

    time.sleep(0.5)  # Respect rate limits

print(f"\n✅ Validated {len(results_cantonese)} English-Cantonese examples!")

📝 Testing judge on 5 English-Cantonese examples...

📌 Example 1: English → Cantonese
   Source: Hello, how are you today?
   Translation: 你好，今日點樣呀？
   ✅ Score: 10.0/10
   📝 Feedback: The translation is excellent, capturing both the meaning and the tone of the source text accurately....

📌 Example 2: English → Cantonese
   Source: I would like to order a cup of coffee.
   Translation: 我想叫一杯咖啡。
   ✅ Score: 9.0/10
   📝 Feedback: The translation provided accurately conveys the meaning of the source text and is fluent. However, t...

📌 Example 3: English → Cantonese
   Source: The weather is beautiful today.
   Translation: 今日天氣好靚。
   ✅ Score: 10.0/10
   📝 Feedback: The translation is excellent, accurately conveying the meaning and tone of the source text. The use ...

📌 Example 4: English → Cantonese
   Source: Can you recommend a good restaurant?
   Translation: 你可唔可以推薦一間好嘅餐廳？
   ✅ Score: 10.0/10
   📝 Feedback: The translation is excellent, accurately conveying the meaning and tone of the

In [ ]:
print("\nTesting judge on 5 English-Mandarin examples...")
print("=" * 80)

en_mandarin_examples = [
    {
        "source": "Good morning, how did you sleep?",
        "translation": "早上好，你睡得好吗？",
        "reference": "早上好，你睡得好吗？",
        "source_lang": "English",
        "target_lang": "Mandarin"
    },
    {
        "source": "I need to catch the train at 3 PM.",
        "translation": "我需要赶下午三点的火车。",
        "reference": "我需要赶下午三点的火车。",
        "source_lang": "English",
        "target_lang": "Mandarin"
    },
    {
        "source": "Could you please speak more slowly?",
        "translation": "你能说慢一点吗？",
        "reference": "你能说慢一点吗？",
        "source_lang": "English",
        "target_lang": "Mandarin"
    },
    {
        "source": "What time does the museum open?",
        "translation": "博物馆几点开门？",
        "reference": "博物馆几点开门？",
        "source_lang": "English",
        "target_lang": "Mandarin"
    },
    {
        "source": "I'm looking for a place to stay tonight.",
        "translation": "我在找今晚住的地方。",
        "reference": "我在找今晚住的地方。",
        "source_lang": "English",
        "target_lang": "Mandarin"
    }
]

results_mandarin = []
for idx, example in enumerate(en_mandarin_examples, 1):
    print(f"\n📌 Example {idx}: English → Mandarin")
    print(f"   Source: {example['source']}")
    print(f"   Translation: {example['translation']}")

    try:
        result = judge(
            source_text=example['source'],
            translated_text=example['translation'],
            reference_text=example['reference'],
            source_lang=example['source_lang'],
            target_lang=example['target_lang']
        )

        print(f"    Score: {result.score:.1f}/10")
        print(f"    Feedback: {result.feedback[:100]}...")

        results_mandarin.append({
            'example': idx,
            'source': example['source'],
            'translation': example['translation'],
            'score': result.score,
            'feedback': result.feedback
        })

    except Exception as e:
        print(f"   ❌ Error: {str(e)}")

    time.sleep(0.5)

print(f"\n Validated {len(results_mandarin)} English-Mandarin examples!")

# Combine results
all_results = results_cantonese + results_mandarin
print(f"\n Total: {len(all_results)} validated examples!")


Testing judge on 5 English-Mandarin examples...

📌 Example 1: English → Mandarin
   Source: Good morning, how did you sleep?
   Translation: 早上好，你睡得好吗？
    Score: 10.0/10
    Feedback: The translation is excellent, accurately conveying the meaning and tone of the source text. The use ...

📌 Example 2: English → Mandarin
   Source: I need to catch the train at 3 PM.
   Translation: 我需要赶下午三点的火车。
    Score: 10.0/10
    Feedback: The translation is excellent, accurately conveying the meaning and intent of the source text. The us...

📌 Example 3: English → Mandarin
   Source: Could you please speak more slowly?
   Translation: 你能说慢一点吗？
    Score: 10.0/10
    Feedback: The translation is excellent, accurately conveying the meaning and tone of the source text. The use ...

📌 Example 4: English → Mandarin
   Source: What time does the museum open?
   Translation: 博物馆几点开门？
    Score: 10.0/10
    Feedback: The translation is accurate, fluent, and culturally appropriate. It correctly uses Mandar

In [ ]:
from statistics import mean, stdev
import numpy as np
import time


class RRWAAggregator:
    """
    Reciprocal-Rank Weighted Aggregation (RRWA)

    Procedure:
    1. Run the judge repeatedly.
    2. Remove scores farther than 2 standard deviations from the mean.
    3. Sort retained scores from highest to lowest.
    4. Assign reciprocal-rank weights: w_r = 1 / r.
    5. Return the weighted score and stability statistics.
    """

    def __init__(self, judge_module, num_runs=10, stability_cv_threshold=0.15):
        self.judge = judge_module
        self.num_runs = num_runs
        self.stability_cv_threshold = stability_cv_threshold

    def aggregate(
        self,
        source_text,
        translated_text,
        reference_text,
        source_lang="English",
        target_lang="Cantonese",
    ):
        scores = []
        feedbacks = []

        print(f"🔄 Running {self.num_runs} independent judge evaluations...")

        for run_idx in range(self.num_runs):
            try:
                result = self.judge(
                    source_text=source_text,
                    translated_text=translated_text,
                    reference_text=reference_text,
                    source_lang=source_lang,
                    target_lang=target_lang,
                )

                scores.append(float(result.score))
                feedbacks.append(str(result.feedback))
                print(f"   Run {run_idx + 1:02d}: {float(result.score):.2f}/10")

            except Exception as error:
                print(
                    f"   Run {run_idx + 1:02d} failed: "
                    f"{type(error).__name__}: {error}"
                )

            time.sleep(0.4)

        if len(scores) != self.num_runs:
            raise RuntimeError(
                f"RRWA required {self.num_runs} successful runs, "
                f"but received {len(scores)}."
            )

        raw_mean = mean(scores)
        raw_std = stdev(scores) if len(scores) > 1 else 0.0

        if raw_std == 0:
            retained_scores = list(scores)
        else:
            retained_scores = [
                score
                for score in scores
                if abs(score - raw_mean) <= 2 * raw_std
            ]

        if not retained_scores:
            raise RuntimeError("All RRWA scores were removed as outliers.")

        ranked_scores = sorted(retained_scores, reverse=True)
        reciprocal_weights = [
            1.0 / rank
            for rank in range(1, len(ranked_scores) + 1)
        ]

        rrwa_score = sum(
            score * weight
            for score, weight in zip(ranked_scores, reciprocal_weights)
        ) / sum(reciprocal_weights)

        retained_mean = mean(retained_scores)
        retained_std = (
            stdev(retained_scores)
            if len(retained_scores) > 1
            else 0.0
        )
        coefficient_variation = (
            retained_std / retained_mean
            if retained_mean > 0
            else float("inf")
        )

        return {
            "rrwa_score": rrwa_score,
            "mean_score": retained_mean,
            "median_score": float(np.median(retained_scores)),
            "std_dev": retained_std,
            "coefficient_variation": coefficient_variation,
            "min_score": min(retained_scores),
            "max_score": max(retained_scores),
            "num_requested_runs": self.num_runs,
            "num_valid_runs": len(scores),
            "num_retained_runs": len(retained_scores),
            "num_outliers_removed": len(scores) - len(retained_scores),
            "is_stable": coefficient_variation < self.stability_cv_threshold,
            "all_scores": scores,
            "retained_scores": retained_scores,
            "feedbacks": feedbacks,
        }


rrwa = RRWAAggregator(judge_module=judge, num_runs=10)
print(" RRWA aggregator initialized")


 RRWA aggregator initialized


In [ ]:
print(" Testing RRWA stability")
print("=" * 80)

test_source = "The food at this restaurant is delicious."
# Deliberately acceptable but not identical to the reference.
test_translation = "呢間餐廳啲嘢食好好味。"
test_reference = "呢間餐廳嘅食物好好味。"

aggregated = rrwa.aggregate(
    source_text=test_source,
    translated_text=test_translation,
    reference_text=test_reference,
    source_lang="English",
    target_lang="Cantonese",
)

print("\nRRWA results")
print(f"RRWA score:               {aggregated['rrwa_score']:.3f}/10")
print(f"Mean score:               {aggregated['mean_score']:.3f}/10")
print(f"Median score:             {aggregated['median_score']:.3f}/10")
print(f"Standard deviation:       {aggregated['std_dev']:.4f}")
print(f"Coefficient of variation: {aggregated['coefficient_variation']:.4f}")
print(f"Outliers removed:         {aggregated['num_outliers_removed']}")
print(f"Valid runs:               {aggregated['num_valid_runs']}")
print(f"Stable:                   {aggregated['is_stable']}")
print(f"All scores:               {aggregated['all_scores']}")

assert aggregated["num_valid_runs"] == 10
assert np.isfinite(aggregated["rrwa_score"])
assert 0.0 <= aggregated["rrwa_score"] <= 10.0
assert aggregated["is_stable"], (
    "The 10-run judge scores exceeded the configured stability threshold."
)

print("\n RRWA produced a stable score across 10 judge runs")


 Testing RRWA stability
🔄 Running 10 independent judge evaluations...
   Run 01: 8.00/10
   Run 02: 8.00/10
   Run 03: 8.00/10
   Run 04: 8.00/10
   Run 05: 8.00/10
   Run 06: 8.00/10
   Run 07: 8.00/10
   Run 08: 8.00/10
   Run 09: 8.00/10
   Run 10: 8.00/10

RRWA results
RRWA score:               8.000/10
Mean score:               8.000/10
Median score:             8.000/10
Standard deviation:       0.0000
Coefficient of variation: 0.0000
Outliers removed:         0
Valid runs:               10
Stable:                   True
All scores:               [8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0]

 RRWA produced a stable score across 10 judge runs


In [ ]:
import dspy
from dspy import Signature, InputField, OutputField, ChainOfThought, Module


class TranslateSignature(Signature):
    """
    Translate English into the requested target language accurately and naturally.
    Cantonese output must use authentic written Cantonese rather than Mandarin
    vocabulary converted into Traditional Chinese characters.
    """

    source_text = InputField(desc="English text to translate")
    source_lang = InputField(desc="Source language")
    target_lang = InputField(desc="Cantonese or Mandarin")
    translation = OutputField(
        desc="Translation only, in the requested target language and script"
    )


class TranslationProgram(Module):
    def __init__(self):
        super().__init__()
        self.translate = ChainOfThought(TranslateSignature)

    def forward(
        self,
        source_text,
        source_lang="English",
        target_lang="Cantonese",
    ):
        return self.translate(
            source_text=source_text,
            source_lang=source_lang,
            target_lang=target_lang,
        )


translation_program = TranslationProgram()

training_examples = [
    dspy.Example(
        source_text="Hello, how are you today?",
        source_lang="English",
        target_lang="Cantonese",
        reference_text="你好，今日點樣呀？",
    ).with_inputs("source_text", "source_lang", "target_lang"),

    dspy.Example(
        source_text="Could you please speak more slowly?",
        source_lang="English",
        target_lang="Cantonese",
        reference_text="你可唔可以講慢啲呀？",
    ).with_inputs("source_text", "source_lang", "target_lang"),

    dspy.Example(
        source_text="Although the deadline was approaching, she remained calm.",
        source_lang="English",
        target_lang="Cantonese",
        reference_text="雖然就快到截止日期，佢仍然保持冷靜。",
    ).with_inputs("source_text", "source_lang", "target_lang"),

    dspy.Example(
        source_text="The meeting has been moved to tomorrow afternoon.",
        source_lang="English",
        target_lang="Mandarin",
        reference_text="会议已改到明天下午举行。",
    ).with_inputs("source_text", "source_lang", "target_lang"),

    dspy.Example(
        source_text="Please check every detail before submitting the report.",
        source_lang="English",
        target_lang="Mandarin",
        reference_text="提交报告前，请检查每一个细节。",
    ).with_inputs("source_text", "source_lang", "target_lang"),
]

assert len(training_examples) == 5
print(" Translation program defined")
print(" Five GEPA examples created")


 Translation program defined
 Five GEPA examples created


In [ ]:
from typing import Optional
import dspy


def translation_metric(
    gold,
    pred,
    trace=None,
    pred_name: Optional[str] = None,
    pred_trace=None,
):
    """
    Return dspy.Prediction(score=float, feedback=str), as required by
    DSPy GEPA when textual feedback is available.
    """

    predicted_translation = getattr(pred, "translation", None)

    if not predicted_translation:
        return dspy.Prediction(
            score=0.0,
            feedback=(
                "The translation program returned no value in the "
                "`translation` output field."
            ),
        )

    try:
        evaluation = judge(
            source_text=gold.source_text,
            translated_text=predicted_translation,
            reference_text=gold.reference_text,
            source_lang=gold.source_lang,
            target_lang=gold.target_lang,
        )

        raw_score = float(evaluation.score)
        normalized_score = max(0.0, min(1.0, raw_score / 10.0))

        feedback = (
            f"Target language: {gold.target_lang}\n"
            f"Source: {gold.source_text}\n"
            f"Candidate: {predicted_translation}\n"
            f"Reference: {gold.reference_text}\n"
            f"Judge score: {raw_score:.1f}/10\n"
            f"Judge feedback: {evaluation.feedback}"
        )

        return dspy.Prediction(
            score=normalized_score,
            feedback=feedback,
        )

    except Exception as error:
        return dspy.Prediction(
            score=0.0,
            feedback=(
                "The translation judge failed. "
                f"{type(error).__name__}: {error}"
            ),
        )


print(" GEPA metric defined")


 GEPA metric defined


In [ ]:

metric_test_example = training_examples[0]

metric_test_prediction = translation_program(
    source_text=metric_test_example.source_text,
    source_lang=metric_test_example.source_lang,
    target_lang=metric_test_example.target_lang,
)

metric_test_result = translation_metric(
    gold=metric_test_example,
    pred=metric_test_prediction,
)

print("Generated translation:")
print(metric_test_prediction.translation)

print("\nMetric score:")
print(metric_test_result.score)

print("\nMetric feedback:")
print(metric_test_result.feedback)

assert isinstance(metric_test_result, dspy.Prediction)
assert hasattr(metric_test_result, "score")
assert hasattr(metric_test_result, "feedback")
assert 0.0 <= float(metric_test_result.score) <= 1.0
assert str(metric_test_result.feedback).strip()
print("\n Metric returns dspy.Prediction(score, feedback)")


Generated translation:
, ?

Metric score:
0.4

Metric feedback:
Target language: Cantonese
Source: Hello, how are you today?
Candidate: , ?
Reference: 你好，今日點樣呀？
Judge score: 4.0/10
Judge feedback: The translation lacks authenticity and cultural appropriateness as it is written in Mandarin instead of Cantonese. To improve, the translator should use genuine Cantonese vocabulary and particles, such as "今日點樣呀" instead of the more formal ". Additionally, attention should be paid to the tone and nuance of the original text, as the reference text conveys a more casual and friendly tone. It is essential to understand the differences between Mandarin and Cantonese, including grammar, vocabulary, and pronunciation, to provide a high-quality translation.

 Metric returns dspy.Prediction(score, feedback)


In [ ]:
import inspect
from pathlib import Path
import dspy

print(" Starting DSPy GEPA optimization")
print("=" * 80)

gepa_trainset = training_examples
assert len(gepa_trainset) == 5

GEPA_LOG_DIR = Path("/content/dspy_gepa_translation/gepa_logs")
GEPA_LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Installed GEPA signature:")
print(inspect.signature(dspy.GEPA))

gepa_optimizer = dspy.GEPA(
    metric=translation_metric,
    reflection_lm=reflection_lm,
    max_full_evals=3,
    reflection_minibatch_size=3,
    candidate_selection_strategy="current_best",
    skip_perfect_score=False,
    use_merge=False,
    num_threads=1,
    failure_score=0.0,
    perfect_score=1.0,
    log_dir=str(GEPA_LOG_DIR),
    track_stats=True,
    seed=42,
)

print("\n GEPA optimizer initialized")
print(" Running compile()...")

optimized_program = gepa_optimizer.compile(
    student=translation_program,
    trainset=gepa_trainset,
)

assert optimized_program is not None

print("\n GEPA optimization iterated and completed without errors")
print(f"Logs saved to: {GEPA_LOG_DIR}")


2026/07/15 22:58:29 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 15 metric calls of the program. This amounts to 3.00 full evals on the train set.
2026/07/15 22:58:29 WARNING dspy.teleprompt.gepa.gepa: No valset provided; Using trainset as valset. This is useful as an inference-time scaling strategy where you want GEPA to find the best solutions for the provided tasks in the trainset, as it makes GEPA overfit prompts to the provided trainset. In order to ensure generalization and perform well on unseen tasks, please provide separate trainset and valset. Provide the smallest valset that is just large enough to match the downstream task distribution, while keeping trainset as large as possible.
2026/07/15 22:58:29 INFO dspy.teleprompt.gepa.gepa: Using 5 examples for tracking Pareto scores.


 Starting DSPy GEPA optimization
Installed GEPA signature:
(metric: dspy.teleprompt.gepa.gepa.GEPAFeedbackMetric, *, auto: Optional[Literal['light', 'medium', 'heavy']] = None, max_full_evals: int | None = None, max_metric_calls: int | None = None, reflection_minibatch_size: int = 3, candidate_selection_strategy: Literal['pareto', 'current_best'] = 'pareto', reflection_lm: dspy.clients.lm.LM | None = None, skip_perfect_score: bool = True, add_format_failure_as_feedback: bool = False, instruction_proposer: 'ProposalFn | None' = None, component_selector: 'ReflectionComponentSelector | str' = 'round_robin', use_merge: bool = True, max_merge_invocations: int | None = 5, num_threads: int | None = None, failure_score: float = 0.0, perfect_score: float = 1.0, log_dir: str | None = None, track_stats: bool = False, use_wandb: bool = False, wandb_api_key: str | None = None, wandb_init_kwargs: dict[str, typing.Any] | None = None, track_best_outputs: bool = False, warn_on_score_mismatch: bool = Tr

GEPA Optimization:   0%|          | 0/15 [00:00<?, ?rollouts/s]2026/07/15 22:58:39 INFO dspy.evaluate.evaluate: Average Metric: 3.3000000000000003 / 5 (66.0%)
2026/07/15 22:58:39 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.6599999999999999 over 5 / 5 examples
GEPA Optimization:  33%|███▎      | 5/15 [00:10<00:21,  2.10s/rollouts]2026/07/15 22:58:39 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.6599999999999999


Average Metric: 1.50 / 3 (50.0%): 100%|██████████| 3/3 [00:04<00:00,  1.52s/it]

2026/07/15 22:58:44 INFO dspy.evaluate.evaluate: Average Metric: 1.5 / 3 (50.0%)


2026/07/15 22:58:47 WARNING dspy.teleprompt.gepa.gepa_utils: The score returned by the metric with pred_name is different from the overall metric score. This can indicate 2 things: Either the metric is non-deterministic (e.g., LLM-as-judge, Semantic score, etc.) or the metric returned a score specific to pred_name that differs from the module level score. Currently, GEPA does not support predictor level scoring (support coming soon), and only requires a feedback text to be provided, which can be specific to the predictor or program level. GEPA will ignore the differing score returned, and instead use module level score. You can safely ignore this warning if using a semantic metric, however, if this mismatch is caused due to predictor scoring, please return module-level scores. To disable this warning, set warn_on_score_mismatch=False.
2026/07/15 22:58:49 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for translate.predict: Translate English into the requested target lan


 GEPA optimization iterated and completed without errors
Logs saved to: /content/dspy_gepa_translation/gepa_logs


In [ ]:
print("\n" + "=" * 80)
print("DSPY + GEPA VALIDATION SUMMARY")
print("=" * 80)

dspy_passed = hasattr(dspy, "GEPA")
judge_passed = (
    len(results_cantonese) >= 5
    and len(results_mandarin) >= 5
    and all(str(item["feedback"]).strip() for item in all_results)
)
rrwa_passed = (
    aggregated["num_valid_runs"] == 10
    and aggregated["is_stable"]
)
gepa_passed = "optimized_program" in globals() and optimized_program is not None

print(f"\n1. DSPy import and GEPA availability: {dspy_passed}")
print(f"2. Five Cantonese + five Mandarin judge examples: {judge_passed}")
print(f"3. RRWA stable across 10 runs: {rrwa_passed}")
print(f"4. Real GEPA compile completed: {gepa_passed}")

all_runtime_checks_passed = all(
    [dspy_passed, judge_passed, rrwa_passed, gepa_passed]
)

print("\n" + "=" * 80)
print(
    "OVERALL STATUS: "
    + (
        "ALL RUNTIME VALIDATIONS PASSED"
        if all_runtime_checks_passed
        else "INCOMPLETE"
    )
)
print("=" * 80)

assert all_runtime_checks_passed, (
    "At least one acceptance criterion has not passed."
)



DSPY + GEPA VALIDATION SUMMARY

1. DSPy import and GEPA availability: True
2. Five Cantonese + five Mandarin judge examples: True
3. RRWA stable across 10 runs: True
4. Real GEPA compile completed: True

OVERALL STATUS: ALL RUNTIME VALIDATIONS PASSED


In [ ]:


from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import dspy

OUTPUT_DIR = Path("/content/dspy_gepa_translation/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

judge_results_path = OUTPUT_DIR / "judge_results.csv"
rrwa_results_path = OUTPUT_DIR / "rrwa_results.json"
environment_path = OUTPUT_DIR / "environment.json"
optimized_program_path = OUTPUT_DIR / "optimized_program.json"

pd.DataFrame(all_results).to_csv(
    judge_results_path,
    index=False,
    encoding="utf-8-sig",
)

rrwa_export = {
    key: value
    for key, value in aggregated.items()
    if key != "feedbacks"
}
rrwa_results_path.write_text(
    json.dumps(rrwa_export, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "runtime": "Google Colab",
    "dspy_version": dspy.__version__,
    "task_model": TASK_MODEL_NAME,
    "judge_model": JUDGE_MODEL_NAME,
    "reflection_model": REFLECTION_MODEL_NAME,
    "gepa_examples": len(training_examples),
    "rrwa_runs": aggregated["num_valid_runs"],
}
environment_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

optimized_program.save(
    str(optimized_program_path),
    save_program=False,
)

print(" Saved:")
print(f"   {judge_results_path}")
print(f"   {rrwa_results_path}")
print(f"   {environment_path}")
print(f"   {optimized_program_path}")

print("\nGitHub publication remains a manual authenticated step:")
print("1. Download this notebook and the results directory.")
print("2. Create an individual GitHub repository.")
print("3. Commit and push the notebook plus generated artifacts.")
print("4. Share the repository URL with the team.")


 Saved:
   /content/dspy_gepa_translation/results/judge_results.csv
   /content/dspy_gepa_translation/results/rrwa_results.json
   /content/dspy_gepa_translation/results/environment.json
   /content/dspy_gepa_translation/results/optimized_program.json

GitHub publication remains a manual authenticated step:
1. Download this notebook and the results directory.
2. Create an individual GitHub repository.
3. Commit and push the notebook plus generated artifacts.
4. Share the repository URL with the team.
